<a href="https://colab.research.google.com/github/ArkhanShimar/HPC/blob/main/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4 - Sobel Edge Detection using CUDA

The flower image is processed using the Gx and Gy Sobel masks and zero padding. The program saves the X gradient, Y gradient and combined edges. It can also accept several PNG filenames.


## Input files


In [1]:
!mkdir -p /content/HPC/Task4
%cd /content/HPC/Task4
!wget -q -O flower.png https://raw.githubusercontent.com/ArkhanShimar/HPC/b22f2d4676028e8e9f66d9c4aa74b010869b8cc5/data/images/flower.png
!wget -q -O lodepng.c https://raw.githubusercontent.com/ArkhanShimar/HPC/b22f2d4676028e8e9f66d9c4aa74b010869b8cc5/Task4/lodepng.c
!wget -q -O lodepng.h https://raw.githubusercontent.com/ArkhanShimar/HPC/b22f2d4676028e8e9f66d9c4aa74b010869b8cc5/Task4/lodepng.h
!wget -q -O LODEPNG_LICENSE.txt https://raw.githubusercontent.com/ArkhanShimar/HPC/b22f2d4676028e8e9f66d9c4aa74b010869b8cc5/Task4/LODEPNG_LICENSE


/content/HPC/Task4


In [2]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


## CUDA source


In [3]:
%%writefile sobel.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <math.h>
#include <cuda_runtime.h>
extern "C" {
#include "lodepng.h"
}

__device__ int grey(const unsigned char *image, long long x, long long y, unsigned width, unsigned height) {
    if (x < 0 || y < 0 || x >= width || y >= height) return 0; // Zero padding.
    size_t p = ((size_t)y * width + (size_t)x)*4;
    return (77*(int)image[p] + 150*(int)image[p+1] + 29*(int)image[p+2]) >> 8;
}
__global__ void sobel_edges(const unsigned char *input, unsigned char *output, unsigned width, unsigned height) {
    size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= (size_t)width*height) return;
    long long x = i % width, y = i / width;
    const int gx[9] = {-1,0,1,-2,0,2,-1,0,1};
    const int gy[9] = {-1,-2,-1,0,0,0,1,2,1};
    int sx = 0, sy = 0;
    for (int dy = -1; dy <= 1; ++dy) {
        for (int dx = -1; dx <= 1; ++dx) {
            int value = grey(input, x+dx, y+dy, width, height);
            int k = (dy+1)*3+dx+1; sx += value*gx[k]; sy += value*gy[k];
        }
    }
    int magnitude = (int)(sqrtf((float)(sx*sx+sy*sy)) + 0.5f);
    int values[3] = {sx < 0 ? -sx : sx, sy < 0 ? -sy : sy, magnitude};
    size_t plane_bytes = (size_t)width*height*4;
    // Keep signed gradients for the magnitude; visualise their absolute values.
    for (int plane = 0; plane < 3; ++plane) {
        unsigned char value = (unsigned char)(values[plane] > 255 ? 255 : values[plane]);
        size_t p = (size_t)plane*plane_bytes+i*4;
        output[p] = output[p+1] = output[p+2] = value;
        output[p+3] = 255;
    }
}
static int checked(cudaError_t code, const char *action) {
    if (code == cudaSuccess) return 1;
    fprintf(stderr, "%s: %s\n", action, cudaGetErrorString(code)); return 0;
}
static int process_image(const char *filename) {
    unsigned char *input = NULL, *output = NULL, *device_in = NULL, *device_out = NULL;
    unsigned width = 0, height = 0, error;
    size_t pixels = 0, bytes = 0, blocks = 0;
    cudaDeviceProp properties;
    int status = 1;
    const char *suffixes[3] = {"_gx.png", "_gy.png", "_edges.png"};
    const char *base = strrchr(filename, '/'); base = base ? base+1 : filename;
    size_t name_length = strlen(base); char *out_name = NULL;
    if (name_length < 5 || strcmp(base+name_length-4, ".png")) { fprintf(stderr, "Expected a .png filename: %s\n", filename); return 1; }
    out_name = (char *)malloc(name_length+7);
    if (!out_name) return 1;
    memcpy(out_name, base, name_length-4); strcpy(out_name+name_length-4, "_edges.png");
    error = lodepng_decode32_file(&input, &width, &height, filename);
    if (error) { fprintf(stderr, "%s: %s\n", filename, lodepng_error_text(error)); goto cleanup; }
    if (!width || !height || (size_t)width > SIZE_MAX/(size_t)height/12) { fprintf(stderr, "Invalid image dimensions.\n"); goto cleanup; }
    pixels = (size_t)width*height; bytes = pixels*4; blocks = (pixels+255)/256;
    output = (unsigned char *)malloc(bytes*3);
    if (!output) goto cleanup;
    if (!checked(cudaGetDeviceProperties(&properties, 0), "GPU properties")) goto cleanup;
    if (blocks > (size_t)properties.maxGridSize[0]) { fprintf(stderr, "Image exceeds grid limit.\n"); goto cleanup; }
    if (!checked(cudaMalloc((void **)&device_in, bytes), "Allocate input image") ||
        !checked(cudaMalloc((void **)&device_out, bytes*3), "Allocate three output images") ||
        !checked(cudaMemcpy(device_in, input, bytes, cudaMemcpyHostToDevice), "Copy image")) goto cleanup;
    sobel_edges<<<(unsigned)blocks, 256>>>(device_in, device_out, width, height);
    if (!checked(cudaGetLastError(), "Launch Sobel kernel") ||
        !checked(cudaDeviceSynchronize(), "Wait for Sobel kernel") ||
        !checked(cudaMemcpy(output, device_out, bytes*3, cudaMemcpyDeviceToHost), "Copy gradient and edge maps")) goto cleanup;
    for (int plane = 0; plane < 3; ++plane) {
        strcpy(out_name+name_length-4, suffixes[plane]);
        error = lodepng_encode32_file(out_name, output+(size_t)plane*bytes, width, height);
        if (error) { fprintf(stderr, "%s: %s\n", out_name, lodepng_error_text(error)); goto cleanup; }
        printf("Saved %s\n", out_name);
    }
    printf("%s -> %s | %ux%u | %zu blocks x 256\n", filename, out_name, width, height, blocks);
    status = 0;
cleanup:
    if (device_in && !checked(cudaFree(device_in), "Free device input")) status = 1;
    if (device_out && !checked(cudaFree(device_out), "Free device output")) status = 1;
    free(input); free(output); free(out_name); return status;
}
int main(int argc, char **argv) {
    if (argc < 2) { fprintf(stderr, "Usage: %s image1.png [image2.png ...]\n", argv[0]); return 1; }
    // Output names use input basenames. Reject collisions before processing.
    for (int i = 1; i < argc; ++i) for (int j = 1; j < i; ++j) {
        const char *a = strrchr(argv[i], '/'), *b = strrchr(argv[j], '/');
        a = a ? a+1 : argv[i]; b = b ? b+1 : argv[j];
        if (!strcmp(a,b)) { fprintf(stderr, "Duplicate image basename: %s\n", a); return 1; }
    }
    int failed = 0;
    for (int i = 1; i < argc; ++i) failed += process_image(argv[i]) != 0;
    printf("Images completed: %d | Failed: %d\n", argc-1-failed, failed);
    return failed ? 1 : 0;
}


Writing sobel.cu


## Compile


In [4]:
!gcc -O2 -DLODEPNG_NO_COMPILE_CPP -c lodepng.c -o lodepng.o
!nvcc -O2 -DLODEPNG_NO_COMPILE_CPP sobel.cu lodepng.o -o sobel


## Run


In [5]:
!./sobel flower.png


Saved flower_gx.png
Saved flower_gy.png
Saved flower_edges.png
flower.png -> flower_edges.png | 4032x6048 | 95256 blocks x 256
Images completed: 1 | Failed: 0


## Output

The output files are saved in `/content/HPC/Task4`.


In [6]:
!ls -lh flower.png flower_gx.png flower_gy.png flower_edges.png


-rw-r--r-- 1 root root  10M Oct  7 18:19 flower_edges.png
-rw-r--r-- 1 root root 9.3M Oct  7 18:19 flower_gx.png
-rw-r--r-- 1 root root 8.5M Oct  7 18:19 flower_gy.png
-rw-r--r-- 1 root root  13M Oct  7 18:18 flower.png
